
<div style="text-align:center; padding:20px; border-radius:12px; background:#eef5ff;">
<h1 style="margin-bottom:6px;">Caminata Aleatoria sobre un grafo</h1>
<h2 style="margin-top:0; margin-bottom:6px;">Simulación II</h2>
<h3 style="margin-top:0;">Estimación del número esperado de 1's en una cadena buena</h3>
</div>

### Objetivo

Implementar una **caminata aleatoria** sobre cadenas binarias buenas para estimar el número esperado de unos en una cadena de longitud \(m=100\).

Una cadena se considera **buena** cuando no contiene dos \(1\)'s consecutivos.



## 1. Formulación del problema

Sea una cadena binaria de longitud \(m\):

$$
x=(x_1,x_2,\ldots,x_m),
\qquad x_i\in\{0,1\}.
$$

La cadena es buena si nunca aparecen dos unos consecutivos. Esto puede escribirse como:

$$
x_i x_{i+1}=0,
\qquad i=1,2,\ldots,m-1.
$$

El conjunto de cadenas buenas es:

$$
\mathcal{G}_m=
\left\{
x\in\{0,1\}^m:
x_i x_{i+1}=0,\;
i=1,\ldots,m-1
\right\}.
$$

Para una cadena \(x\), definimos:

$$
N_1(x)=\sum_{i=1}^{m}x_i,
$$

donde \(N_1(x)\) es la cantidad de \(1\)'s de la cadena.

El objetivo es estimar:

$$
\mu=E[N_1(X)].
$$



## 2. Método de la caminata aleatoria

La simulación comienza con:

$$
X_0=(0,0,\ldots,0),
$$

que siempre es una cadena válida.

En cada paso se selecciona una posición al azar:

$$
I_t\sim \text{Uniforme}\{1,2,\ldots,m\}.
$$

Después se intenta cambiar el bit elegido:

$$
0\longleftrightarrow 1.
$$

Si la nueva cadena propuesta \(Y_t\) sigue siendo buena, el cambio se conserva. Si aparecen dos unos consecutivos, la propuesta se rechaza y la cadena se queda como estaba:

$$
X_{t+1}=
\begin{cases}
Y_t, & \text{si }Y_t\in\mathcal{G}_m,\\
X_t, & \text{si }Y_t\notin\mathcal{G}_m.
\end{cases}
$$



## 3. Periodo de calentamiento y estimador

Antes de calcular el promedio se realizan \(B=2000\) pasos de calentamiento. Estos pasos sirven para disminuir la influencia de la cadena inicial.

Después se realizan:

$$
T=100000
$$

pasos de simulación.

En cada paso se cuenta el número de unos:

$$
N_1(X_t)=\sum_{i=1}^{m}X_{t,i}.
$$

El estimador utilizado es:

$$
\widehat{\mu}
=
\frac{1}{T}
\sum_{t=1}^{T}N_1(X_t).
$$

Por lo tanto,

$$
\widehat{\mu}\approx E[N_1(X)].
$$



## 4. Comentarios sobre la sintaxis de Python

- `import random` carga las herramientas de aleatoriedad.
- `def` define una función.
- `for` repite un bloque de instrucciones.
- `range()` genera los valores utilizados en el ciclo.
- `len()` obtiene la longitud de una lista.
- `if` permite evaluar una condición.
- `and` requiere que dos condiciones sean verdaderas.
- `return` devuelve el resultado de una función.
- `[0] * m` crea una lista con `m` ceros.
- `random.randint(0, m-1)` elige una posición aleatoria.
- `sum(estado)` cuenta los \(1\)'s porque la cadena solo contiene ceros y unos.
- `f"..."` permite insertar valores dentro de un texto.


In [1]:
import random

def es_buena(secuencia):
    """Verifica que no existan 1's adyacentes."""

    # Recorremos la cadena hasta la penúltima posición.
    for i in range(len(secuencia) - 1):

        # Si encontramos dos 1's consecutivos, la cadena no es buena.
        if secuencia[i] == 1 and secuencia[i + 1] == 1:
            return False

    # Si terminó el ciclo sin encontrar 11, la cadena sí es válida.
    return True


def simular_cantidad_unos(m=100, num_pasos=100000, periodo_calentamiento=2000):

    # Empezamos con una cadena válida formada por m ceros.
    estado = [0] * m

    # -----------------------------
    # PERIODO DE CALENTAMIENTO
    # -----------------------------
    # Estos estados no se usan en el promedio final.
    for _ in range(periodo_calentamiento):

        # randint selecciona una posición al azar entre 0 y m-1.
        idx = random.randint(0, m - 1)

        # Si el bit es 1, lo cambiamos a 0.
        if estado[idx] == 1:
            estado[idx] = 0

        else:
            # Si es 0, intentamos cambiarlo a 1.
            estado[idx] = 1

            # Si la cadena deja de ser buena, rechazamos el cambio.
            if not es_buena(estado):
                estado[idx] = 0

    total_unos = 0
    min_unos = m
    max_unos = 0

    # -----------------------------
    # SIMULACIÓN PRINCIPAL
    # -----------------------------
    for _ in range(num_pasos):

        idx = random.randint(0, m - 1)

        if estado[idx] == 1:
            estado[idx] = 0

        else:
            estado[idx] = 1

            if not es_buena(estado):
                estado[idx] = 0

        # Como la cadena solo tiene ceros y unos,
        # sum(estado) es exactamente la cantidad actual de 1's.
        unos_actuales = sum(estado)

        total_unos += unos_actuales

        if unos_actuales < min_unos:
            min_unos = unos_actuales

        if unos_actuales > max_unos:
            max_unos = unos_actuales

    # Promedio Monte Carlo del número de 1's.
    promedio_unos = total_unos / num_pasos

    return promedio_unos, min_unos, max_unos



## 5. Ejecución para \(m=100\)

Se utilizan los valores:

$$
m=100,\qquad T=100000,\qquad B=2000.
$$

El programa devuelve el promedio estimado, además del mínimo y máximo número de unos observados.


In [2]:
# ==========================================
# EJECUCIÓN PARA m = 100
# ==========================================

m = 100
pasos = 100000

promedio, minimo, maximo = simular_cantidad_unos(
    m=m,
    num_pasos=pasos
)

print(f"--- Simulación MCMC para m = {m} ---")
print(f"Promedio de 1's en las cadenas buenas (μ): {promedio:.2f}")
print(f"Mínimo de 1's observado: {minimo}")
print(f"Máximo de 1's observado: {maximo}")


--- Simulación MCMC para m = 100 ---
Promedio de 1's en las cadenas buenas (μ): 27.71
Mínimo de 1's observado: 15
Máximo de 1's observado: 38



## 6. Interpretación

El valor:

$$
\widehat{\mu}
=
\frac{1}{100000}
\sum_{t=1}^{100000}N_1(X_t)
$$

representa la estimación del **número promedio de \(1\)'s que se espera encontrar en una cadena buena de longitud 100**.

El mínimo y el máximo solo muestran los valores extremos observados durante la caminata.



## 7. Solución final

<div style="padding:14px; border-left:6px solid #2e7d32; background:#eef8ef; border-radius:8px;">

La caminata aleatoria recorre únicamente cadenas buenas, ya que cualquier cambio que genere dos \(1\)'s consecutivos se rechaza.

La estimación final se calcula con:

$$
\boxed{
\widehat{\mu}
=
\frac{1}{T}
\sum_{t=1}^{T}
\sum_{i=1}^{m}X_{t,i}
}
$$

Para \(m=100\) y \(T=100000\), el valor mostrado por Python como **Promedio de 1's** es la estimación solicitada.

</div>
